# 05 — Interactive MNIST Digit Recognizer (Mini Project)

**Objective:** Build an end-to-end digit recognition application.
This notebook trains the MNIST model and documents the preprocessing pipeline used by the interactive Streamlit app (`app.py`).

## Architecture
- **This notebook** (`05_mnist_digit_recognizer.ipynb`): Trains the model (if not already saved) and demonstrates the full preprocessing pipeline.
- **`app.py`**: A Streamlit web app with an HTML5 canvas where users can draw digits and get live predictions.

---
## 1. Imports and Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Path to saved model — relative to this notebook's directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if '__file__' in dir() else os.getcwd()
MODEL_PATH   = os.path.join(NOTEBOOK_DIR, "mnist_model.keras")

print(f"TensorFlow : {tf.__version__}")
print(f"Model path : {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")

---
## 2. Train or Load Model

In [ ]:
def train_mnist_model(save_path):
    """Train a 3-Dense-layer model on MNIST and save it."""
    print("Loading MNIST...")
    (X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
    X_train = X_train_raw.astype("float32") / 255.0
    X_test  = X_test_raw.astype("float32")  / 255.0

    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64,  activation="relu"),
        keras.layers.Dense(10,  activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    print("Training model (10 epochs)...")
    model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test accuracy: {test_acc * 100:.2f}%")

    model.save(save_path)
    print(f"Model saved: {save_path}")
    return model, (X_test, y_test)

if os.path.exists(MODEL_PATH):
    print("Loading existing model...")
    model = keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully.")
    (_, _), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
    X_test = X_test_raw.astype("float32") / 255.0
else:
    print("No saved model found. Training a new one...")
    model, (X_test, y_test) = train_mnist_model(MODEL_PATH)

model.summary()

---
## 3. Image Preprocessing Pipeline

When a user draws on the canvas in `app.py`, the raw image must be processed to match MNIST's format before feeding it to the model. MNIST images are:
- **28 x 28 pixels**
- **Grayscale**
- **Dark background (0), light digit (255)**
- **Pixel values normalized to [0, 1]**

In [ ]:
def preprocess_canvas_image(pil_image):
    """
    Preprocess a PIL image drawn on a canvas into a 28x28 MNIST-compatible array.

    Pipeline:
    1. Convert to RGBA (handle transparency)
    2. Composite on white background to handle transparency
    3. Convert to grayscale
    4. Invert: canvas is white bg + dark stroke -> MNIST is dark bg + white digit
    5. Threshold and crop bounding box of non-zero pixels
    6. Preserve aspect ratio while fitting into a 20x20 box
    7. Center the digit on a 28x28 canvas with 4px padding
    8. Normalize to [0, 1]
    9. Reshape to (1, 28, 28) for model input

    Returns:
        processed (np.ndarray): shape (1, 28, 28), dtype float32
        debug_img (np.ndarray): shape (28, 28) for visualization
        is_blank (bool): True if the canvas appears to be blank
    """
    # Step 1 & 2: Handle transparency
    if pil_image.mode in ('RGBA', 'LA'):
        background = Image.new('RGB', pil_image.size, (255, 255, 255))
        if pil_image.mode == 'RGBA':
            background.paste(pil_image, mask=pil_image.split()[3])
        else:
            background.paste(pil_image, mask=pil_image.split()[1])
        pil_image = background
    
    # Step 3: Convert to grayscale
    gray = pil_image.convert("L")
    img_array = np.array(gray, dtype=np.float32)

    # Step 4: Invert polarity (white bg + black drawing -> black bg + white digit)
    img_array = 255.0 - img_array

    # Step 5: Check if blank
    if img_array.max() < 20:
        return None, np.zeros((28, 28), dtype=np.float32), True

    # Step 5 cont: Crop bounding box
    threshold = 20
    rows = np.any(img_array > threshold, axis=1)
    cols = np.any(img_array > threshold, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    cropped = img_array[rmin:rmax+1, cmin:cmax+1]

    # Step 6: Preserve aspect ratio and resize into 20x20
    h, w = cropped.shape
    scale = 20.0 / max(h, w)
    new_h = max(1, int(h * scale))
    new_w = max(1, int(w * scale))
    pil_cropped = Image.fromarray(cropped.astype(np.uint8))
    pil_resized = pil_cropped.resize((new_w, new_h), Image.LANCZOS)
    resized = np.array(pil_resized, dtype=np.float32)

    # Step 7: Initial placement on 28x28 canvas
    canvas_28 = np.zeros((28, 28), dtype=np.float32)
    r_offset = (28 - new_h) // 2
    c_offset = (28 - new_w) // 2
    canvas_28[r_offset:r_offset+new_h, c_offset:c_offset+new_w] = resized

    # Step 8: Center of Mass translation to (14.0, 14.0) (MNIST standard)
    from scipy.ndimage import center_of_mass, shift
    cy, cx = center_of_mass(canvas_28)
    if not np.isnan(cy) and not np.isnan(cx):
        shift_y = 14.0 - cy
        shift_x = 14.0 - cx
        canvas_28 = shift(canvas_28, (shift_y, shift_x), order=1, mode='constant', cval=0.0)

    # Step 9: Normalize to [0, 1]
    canvas_28 = np.clip(canvas_28, 0.0, 255.0) / 255.0

    # Step 10: Reshape for model
    processed = canvas_28.reshape(1, 28, 28)
    return processed, canvas_28, False



---
## 4. Test the Preprocessing Pipeline with MNIST Samples

We test the pipeline using real MNIST test images converted to PIL and back, simulating what would come from a canvas.

In [ ]:
# Test preprocessing on MNIST samples
# Simulate canvas: invert MNIST (dark digit on white background)

test_indices = [np.where(y_test == d)[0][0] for d in range(10)]

fig, axes = plt.subplots(3, 10, figsize=(16, 5))

for col, idx in enumerate(test_indices):
    original = X_test[idx]            # shape (28, 28), values [0, 1]
    true_label = int(y_test[idx])

    # Simulate canvas: upscale to 200x200 and invert polarity
    orig_pil = Image.fromarray((original * 255).astype(np.uint8))
    canvas_pil = Image.fromarray((255 - original * 255).astype(np.uint8))  # inverted
    canvas_pil_large = canvas_pil.resize((200, 200), Image.NEAREST)

    # Run preprocessing
    processed, debug_img, is_blank = preprocess_canvas_image(canvas_pil_large)

    if not is_blank:
        probs = model.predict(processed, verbose=0)[0]
        pred  = int(np.argmax(probs))
        conf  = probs[pred] * 100
    else:
        pred, conf = -1, 0.0

    # Row 0: original MNIST
    axes[0, col].imshow(original, cmap="gray")
    axes[0, col].set_title(f"True: {true_label}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: debug (preprocessed)
    axes[1, col].imshow(debug_img, cmap="gray")
    axes[1, col].set_title("Preprocessed", fontsize=7)
    axes[1, col].axis("off")

    # Row 2: prediction
    color = "green" if pred == true_label else "red"
    axes[2, col].imshow(debug_img, cmap="gray")
    axes[2, col].set_title(f"Pred: {pred}\n{conf:.1f}%", fontsize=8, color=color)
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel("Preprocessed", fontsize=9)
axes[2, 0].set_ylabel("Prediction", fontsize=9)

plt.suptitle("Preprocessing Pipeline Test — One Sample per Digit", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Single Digit Prediction Demo

In [ ]:
def predict_and_display(pil_image, true_label=None):
    """Run preprocessing and display prediction with confidence scores."""
    processed, debug_img, is_blank = preprocess_canvas_image(pil_image)

    if is_blank:
        print("Canvas appears blank. Please draw a digit first.")
        return

    probs = model.predict(processed, verbose=0)[0]
    pred  = int(np.argmax(probs))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.imshow(debug_img, cmap="gray")
    ax1.set_title(f"Preprocessed Input (28x28)")
    ax1.axis("off")

    bars = ax2.bar(range(10), probs * 100, color=["tomato" if i == pred else "steelblue" for i in range(10)])
    ax2.set_xticks(range(10))
    ax2.set_xlabel("Digit Class")
    ax2.set_ylabel("Confidence (%)")
    ax2.set_title(f"Prediction: {pred}  (Confidence: {probs[pred]*100:.1f}%)")
    ax2.set_ylim(0, 105)
    ax2.grid(axis="y", linestyle="--", alpha=0.4)

    if true_label is not None:
        correct = pred == true_label
        fig.suptitle(f"True label: {true_label} | Predicted: {pred} | {'CORRECT' if correct else 'WRONG'}",
                     color="green" if correct else "red", fontsize=13)

    plt.tight_layout()
    plt.show()

    print("\nClass probabilities:")
    for digit, prob in enumerate(probs):
        bar = "#" * int(prob * 40)
        print(f"  Digit {digit}: {prob * 100:5.1f}%  {bar}")

# Demo on digit '7'
idx_7 = np.where(y_test == 7)[0][0]
sample_7_pil = Image.fromarray((255 - X_test[idx_7] * 255).astype(np.uint8)).resize((200, 200))
predict_and_display(sample_7_pil, true_label=7)

---
## 6. How to Launch the Interactive App

The interactive drawing interface is in `app.py` in this folder.

### Setup
```bash
cd Week-1
pip install -r requirements.txt
```

### Launch
```bash
streamlit run app.py
```

This opens a browser tab at `http://localhost:8501` where you can:
1. Draw a digit on the canvas with your mouse.
2. Click **Predict** to see the model's prediction and confidence scores.
3. Click **Clear** to reset the canvas.

The app loads `mnist_model.keras` from the same directory. If the model file is missing, run this notebook first to generate it.

---
## 7. Model Confidence Warning

> **Important:** Model confidence (softmax probability) is not a guarantee of correctness.
> A model can be 99% confident in an incorrect prediction, especially when the drawn digit:
> - Is positioned differently from MNIST training images.
> - Is drawn with unusual stroke thickness.
> - Contains background noise or partial strokes.
>
> The preprocessing pipeline attempts to normalize these differences, but some variance is unavoidable.